# RQ4 — Repository-Level Cross-Project Evaluation

This notebook reproduces the statistical analysis of the **RQ4 repository-level cross-project evaluation** from the project-level Macro-F1 scores stored locally in `../metrics/evaluation_macro_f1.csv`.

The evaluation contains 30 repositories and six approaches evaluated under the common 11-class protocol. The final evaluation inputs use the frozen aggregation rules adopted by the study: TF-IDF + Linear SVM uses its final majority-vote predictions; BiLSTM and CodeBERT use their adaptive first-to-two predictions; GPT-OSS, FLACCS, and CodeFuse HQCM use the corrected strict consensus reconstructed from the audited generative outputs.

Projects are the paired experimental units and approaches are the within-project conditions. The inferential workflow follows the prespecified repeated-measures decision rule:

1. inspect Shapiro–Wilk normality for every approach;
2. if normality is not rejected for all approaches, use repeated-measures ANOVA, applying the Greenhouse–Geisser correction when Mauchly's test rejects sphericity;
3. otherwise, use the Friedman test;
4. run pairwise post-hoc tests only after a significant omnibus result;
5. use paired Student's $t$-tests with Hedges' $g$ on the parametric route, or Wilcoxon signed-rank tests with matched-pairs rank-biserial correlation ($r_{rb}$) on the non-parametric route;
6. adjust the complete family of pairwise $p$-values with Holm's procedure.

The notebook is intentionally local-only: it performs no network downloads and installs no packages at runtime. Required Python dependencies should be installed from the replication package environment before execution.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import pingouin as pg
except ImportError as exc:
    raise ImportError(
        "This notebook requires 'pingouin'. Install the replication-package "
        "dependencies before executing the analysis."
    ) from exc

DATA_FILENAME = "evaluation_macro_f1.csv"
EXPECTED_PROJECTS = 30
EXPECTED_APPROACH_COLUMNS = [
    "svm",
    "codebert",
    "bilstm",
    "gpt-oss",
    "flaccs",
    "hqcm",
]

DISPLAY_LABELS = {
    "svm": "TF-IDF + Linear SVM",
    "codebert": "CodeBERT",
    "bilstm": "BiLSTM",
    "gpt-oss": "GPT-OSS 20B",
    "flaccs": "FLACCS",
    "hqcm": "CodeFuse HQCM",
}


def locate_evaluation_file(filename=DATA_FILENAME):
    """Locate the frozen evaluation input from either the notebook folder or repo root."""
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / ".." / "metrics" / filename,
        cwd / "metrics" / filename,
        cwd / "results" / "rq4_1" / "cross_project" / "metrics" / filename,
    ]

    for parent in (cwd, *cwd.parents):
        candidates.append(
            parent / "results" / "rq4_1" / "cross_project" / "metrics" / filename
        )

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.is_file():
            return candidate

    checked = "\n".join(f"  - {path}" for path in seen)
    raise FileNotFoundError(
        f"Could not locate {filename}. Checked:\n{checked}"
    )


DATA_PATH = locate_evaluation_file()

# The CSV is a machine-readable file: comma separator and decimal point.
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()

expected_columns = ["project", *EXPECTED_APPROACH_COLUMNS]
missing_columns = [c for c in expected_columns if c not in df_raw.columns]
extra_columns = [c for c in df_raw.columns if c not in expected_columns]
if missing_columns or extra_columns:
    raise ValueError(
        "Unexpected evaluation schema. "
        f"Missing={missing_columns}; extra={extra_columns}."
    )

if len(df_raw) != EXPECTED_PROJECTS:
    raise ValueError(
        f"Expected {EXPECTED_PROJECTS} projects, found {len(df_raw)}."
    )

if df_raw["project"].isna().any() or (df_raw["project"].astype(str).str.strip() == "").any():
    raise ValueError("Project identifiers must be non-empty.")

if df_raw["project"].duplicated().any():
    duplicates = df_raw.loc[df_raw["project"].duplicated(), "project"].astype(str).tolist()
    raise ValueError(f"Duplicate project identifiers were found: {duplicates}")

for column in EXPECTED_APPROACH_COLUMNS:
    df_raw[column] = pd.to_numeric(df_raw[column], errors="raise")

if df_raw[EXPECTED_APPROACH_COLUMNS].isna().any().any():
    raise ValueError(
        "The cross-project evaluation must contain complete paired scores for all 30 projects."
    )

invalid_mask = (
    (df_raw[EXPECTED_APPROACH_COLUMNS] < 0)
    | (df_raw[EXPECTED_APPROACH_COLUMNS] > 1)
)
if invalid_mask.any().any():
    raise ValueError("Evaluation Macro-F1 values must lie in the [0, 1] interval.")

# Keep the frozen model order from the evaluation definition.
df_clean = (
    df_raw.set_index("project")[EXPECTED_APPROACH_COLUMNS]
    .rename(columns=DISPLAY_LABELS)
    .copy()
)

print(f"Evaluation file: {DATA_PATH}")
print(f"Paired projects: {len(df_clean)}")
print(f"Approaches ({len(df_clean.columns)}): {df_clean.columns.tolist()}")
display(df_clean.head())


## Automatic repeated-measures analysis

The compact evaluation table reports mean project Macro-F1, standard deviation, an unadjusted 95% confidence interval, and mean within-project rank. Lower mean ranks indicate better average relative performance.

The input is frozen to the six internally comparable RQ4 approaches. The notebook intentionally fails if a project, approach, or score is missing instead of silently changing the paired cohort.

“Not statistically distinguishable from the leading approach” is not an equivalence claim. Results from original related-work datasets, incompatible class subsets, or different validation protocols must be analyzed separately.


In [ ]:
import textwrap
from itertools import combinations
from scipy.stats import friedmanchisquare, shapiro, t, ttest_rel


def _format_decimal(value, digits=3):
    text = f"{float(value):.{digits}f}"
    if text.startswith("0."):
        return text[1:]
    if text.startswith("-0."):
        return "-" + text[2:]
    return text


def _format_p_markdown(p_value):
    p_value = float(p_value)
    if p_value < 0.001:
        return r"$p<.001$"
    return f"$p={_format_decimal(p_value)}$"


def _format_p_plain(p_value, digits=4):
    p_value = float(p_value)
    if p_value < 0.001:
        return "<.001"
    return f"{p_value:.{digits}f}".lstrip("0")


def _effect_magnitude(effect, route):
    value = abs(float(effect))

    if route == "parametric":
        if value < 0.2:
            return "Negligible"
        if value < 0.5:
            return "Small"
        if value < 0.8:
            return "Medium"
        return "Large"

    if value < 0.1:
        return "Negligible"
    if value < 0.3:
        return "Small"
    if value < 0.5:
        return "Medium"
    return "Large"


def _sphericity_values(result):
    """Support named and tuple-like Pingouin return objects."""
    names = ("spher", "W", "chi2", "dof", "pval")
    if all(hasattr(result, name) for name in names):
        return {name: getattr(result, name) for name in names}

    return dict(zip(names, result))


def _pingouin_value(row, *candidates):
    normalized = {
        str(column).lower().replace("-", "_"): column
        for column in row.index
    }
    for candidate in candidates:
        key = candidate.lower().replace("-", "_")
        if key in normalized:
            return row[normalized[key]]
    raise KeyError(
        f"None of these columns were found: {candidates}"
    )


def _descriptive_table(data):
    n = len(data)
    means = data.mean()
    standard_deviations = data.std(ddof=1)
    standard_errors = standard_deviations / np.sqrt(n)
    critical_t = t.ppf(0.975, df=n - 1)
    mean_ranks = data.rank(
        axis=1,
        method="average",
        ascending=False,
    ).mean()

    table = pd.DataFrame({
        "N": n,
        "Mean Macro-F1": means,
        "SD": standard_deviations,
        "CI95 Lower": means - critical_t * standard_errors,
        "CI95 Upper": means + critical_t * standard_errors,
        "Mean Rank": mean_ranks,
    })

    return table.sort_values(
        ["Mean Rank", "Mean Macro-F1"],
        ascending=[True, False],
    )


def _pairwise_tests(data, route, alpha):
    rows = []

    for model_a, model_b in combinations(data.columns, 2):
        values_a = data[model_a].to_numpy()
        values_b = data[model_b].to_numpy()

        if route == "parametric":
            statistic, p_value = ttest_rel(values_a, values_b)

            if np.allclose(values_a - values_b, 0):
                effect = 0.0
            else:
                effect = pg.compute_effsize(
                    values_a,
                    values_b,
                    paired=True,
                    eftype="hedges",
                )

            statistic_name = "t"
            effect_name = "Hedges' g"
        else:
            if np.allclose(values_a - values_b, 0):
                statistic = 0.0
                p_value = 1.0
                effect = 0.0
            else:
                wilcoxon = pg.wilcoxon(
                    values_a,
                    values_b,
                    alternative="two-sided",
                ).iloc[0]
                statistic = _pingouin_value(
                    wilcoxon,
                    "W_val",
                    "W-val",
                )
                p_value = _pingouin_value(
                    wilcoxon,
                    "p_val",
                    "p-val",
                )
                effect = _pingouin_value(wilcoxon, "RBC")

            statistic_name = "W"
            effect_name = "Rank-biserial correlation"

        rows.append({
            "A": model_a,
            "B": model_b,
            "Mean A": float(np.mean(values_a)),
            "Mean B": float(np.mean(values_b)),
            "Statistic name": statistic_name,
            "Statistic": float(statistic),
            "p_raw": float(p_value),
            "Effect name": effect_name,
            "Effect": float(effect),
        })

    pairwise = pd.DataFrame(rows)
    reject, corrected_p = pg.multicomp(
        pairwise["p_raw"].to_numpy(),
        alpha=alpha,
        method="holm",
    )
    pairwise["p_Holm"] = corrected_p
    pairwise["Significant"] = reject
    pairwise["Winner"] = np.where(
        pairwise["Effect"] >= 0,
        pairwise["A"],
        pairwise["B"],
    )
    pairwise["Loser"] = np.where(
        pairwise["Effect"] >= 0,
        pairwise["B"],
        pairwise["A"],
    )
    pairwise["Absolute effect"] = pairwise["Effect"].abs()
    pairwise["Effect magnitude"] = pairwise["Effect"].map(
        lambda value: _effect_magnitude(value, route)
    )

    return pairwise


def run_model_comparison(data, alpha=0.05):
    descriptive = _descriptive_table(data)

    shapiro_rows = []
    for model in data.columns:
        statistic, p_value = shapiro(data[model])
        shapiro_rows.append({
            "Model": model,
            "N": int(data[model].notna().sum()),
            "W": float(statistic),
            "p": float(p_value),
            "Normality not rejected": bool(p_value >= alpha),
        })

    shapiro_results = pd.DataFrame(shapiro_rows).set_index("Model")
    all_normal = bool(
        shapiro_results["Normality not rejected"].all()
    )
    route = "parametric" if all_normal else "non-parametric"

    n_projects = len(data)
    n_models = len(data.columns)
    sphericity = None

    if route == "parametric":
        long_data = (
            data.rename_axis("Project")
            .reset_index()
            .melt(
                id_vars="Project",
                var_name="Model",
                value_name="Macro-F1",
            )
        )

        sphericity = _sphericity_values(
            pg.sphericity(
                data=long_data,
                dv="Macro-F1",
                within="Model",
                subject="Project",
            )
        )

        anova = pg.rm_anova(
            data=long_data,
            dv="Macro-F1",
            within="Model",
            subject="Project",
            correction=True,
            detailed=True,
            effsize="ng2",
        )
        anova_row = anova.iloc[0]
        statistic = float(_pingouin_value(anova_row, "F"))
        p_uncorrected = float(
            _pingouin_value(anova_row, "p_unc", "p-unc")
        )
        epsilon = float(_pingouin_value(anova_row, "eps"))
        effect = float(_pingouin_value(anova_row, "ng2"))

        if float(sphericity["pval"]) < alpha:
            p_value = float(
                _pingouin_value(
                    anova_row,
                    "p_GG_corr",
                    "p-GG-corr",
                )
            )
            df1 = epsilon * (n_models - 1)
            df2 = epsilon * (n_models - 1) * (n_projects - 1)
            correction = "Greenhouse-Geisser"
        else:
            p_value = p_uncorrected
            df1 = n_models - 1
            df2 = (n_models - 1) * (n_projects - 1)
            correction = "None"

        omnibus = {
            "test": "Repeated-measures ANOVA",
            "statistic name": "F",
            "statistic": statistic,
            "df1": float(df1),
            "df2": float(df2),
            "p": p_value,
            "effect name": "eta_G_squared",
            "effect": effect,
            "epsilon": epsilon,
            "correction": correction,
            "significant": bool(p_value < alpha),
        }
    else:
        statistic, p_value = friedmanchisquare(
            *[data[column].to_numpy() for column in data.columns]
        )
        kendalls_w = statistic / (
            n_projects * (n_models - 1)
        )
        omnibus = {
            "test": "Friedman",
            "statistic name": "chi-square",
            "statistic": float(statistic),
            "df1": float(n_models - 1),
            "df2": np.nan,
            "p": float(p_value),
            "effect name": "Kendall's W",
            "effect": float(kendalls_w),
            "epsilon": np.nan,
            "correction": "Not applicable",
            "significant": bool(p_value < alpha),
        }

    if omnibus["significant"]:
        pairwise = _pairwise_tests(data, route, alpha)
        significant_pairwise = pairwise.loc[
            pairwise["Significant"]
        ].copy()
    else:
        pairwise = None
        significant_pairwise = None

    # The leading approach is defined by the primary performance metric.
    leader = descriptive["Mean Macro-F1"].idxmax()
    not_distinguishable = [leader]

    if pairwise is not None:
        for model in descriptive.index[1:]:
            comparison = pairwise.loc[
                (
                    (pairwise["A"] == leader)
                    & (pairwise["B"] == model)
                )
                | (
                    (pairwise["A"] == model)
                    & (pairwise["B"] == leader)
                )
            ]
            if len(comparison) != 1:
                raise ValueError(
                    "Missing or duplicated leader comparison for "
                    f"{model}."
                )
            if not bool(comparison.iloc[0]["Significant"]):
                not_distinguishable.append(model)

    summary = descriptive.copy()
    summary["Not distinguishable from leader"] = [
        model in not_distinguishable
        for model in summary.index
    ]

    print("DESCRIPTIVE APPROACH TABLE")
    display(summary.round({
        "Mean Macro-F1": 4,
        "SD": 4,
        "CI95 Lower": 4,
        "CI95 Upper": 4,
        "Mean Rank": 2,
    }))

    print("SHAPIRO-WILK TESTS")
    shapiro_display = shapiro_results.copy()
    shapiro_display["W"] = shapiro_display["W"].round(4)
    shapiro_display["p"] = shapiro_display["p"].map(
        _format_p_plain
    )
    display(shapiro_display)

    print(f"SELECTED ROUTE: {route}")
    display(pd.DataFrame([omnibus]))

    if pairwise is None:
        print(
            "The omnibus test was not significant; confirmatory "
            "post-hoc comparisons were not performed."
        )
    else:
        print(
            "SIGNIFICANT HOLM-CORRECTED PAIRWISE COMPARISONS "
            f"({len(significant_pairwise)} of {len(pairwise)})"
        )
        pairwise_display = significant_pairwise.copy()
        pairwise_display["p_raw"] = pairwise_display["p_raw"].map(
            _format_p_plain
        )
        pairwise_display["p_Holm"] = pairwise_display[
            "p_Holm"
        ].map(_format_p_plain)
        display(pairwise_display.round({
            "Mean A": 4,
            "Mean B": 4,
            "Statistic": 4,
            "Effect": 4,
        }))

    return {
        "data": data,
        "descriptive": descriptive,
        "summary": summary,
        "shapiro": shapiro_results,
        "route": route,
        "sphericity": sphericity,
        "omnibus": omnibus,
        "pairwise": pairwise,
        "significant_pairwise": significant_pairwise,
        "leader": leader,
        "not_distinguishable": not_distinguishable,
        "alpha": alpha,
    }


def generate_model_comparison_report(
    analysis,
    title="RQ4 internal evaluation interpretation generated from current data",
):
    route = analysis["route"]
    shapiro_results = analysis["shapiro"]
    omnibus = analysis["omnibus"]
    significant = analysis["significant_pairwise"]
    pairwise = analysis["pairwise"]
    leader = analysis["leader"]
    descriptive = analysis["descriptive"]
    not_distinguishable = analysis["not_distinguishable"]

    if route == "non-parametric":
        failed_models = shapiro_results.index[
            ~shapiro_results["Normality not rejected"]
        ].tolist()
        assumption_text = (
            "Shapiro–Wilk rejected normality for at least one model "
            "(" + ", ".join(failed_models) + "). The non-parametric "
            "repeated-measures route was therefore selected."
        )
        omnibus_text = (
            "The Friedman test "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall approach effect, "
            + f"$\\chi^2({int(omnibus['df1'])})="
            + f"{omnibus['statistic']:.2f}$, "
            + f"{_format_p_markdown(omnibus['p'])}, "
            + f"Kendall's $W={_format_decimal(omnibus['effect'])}$."
        )
        posthoc_name = (
            "Wilcoxon signed-rank tests with Holm correction and "
            "matched-pairs rank-biserial correlations"
        )
    else:
        assumption_text = (
            "Shapiro–Wilk did not reject normality for any approach, "
            "so the parametric repeated-measures route was selected."
        )
        sphericity = analysis["sphericity"]
        if float(sphericity["pval"]) < analysis["alpha"]:
            assumption_text += (
                " Mauchly's test rejected sphericity "
                f"($W={_format_decimal(sphericity['W'])}$, "
                f"{_format_p_markdown(sphericity['pval'])}); the "
                "Greenhouse–Geisser correction was applied."
            )
        else:
            assumption_text += (
                " Mauchly's test did not reject sphericity "
                f"($W={_format_decimal(sphericity['W'])}$, "
                f"{_format_p_markdown(sphericity['pval'])})."
            )

        omnibus_text = (
            "The repeated-measures ANOVA "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall approach effect, "
            + f"$F({omnibus['df1']:.2f},{omnibus['df2']:.2f})="
            + f"{omnibus['statistic']:.2f}$, "
            + f"{_format_p_markdown(omnibus['p'])}, "
            + f"$\\eta_G^2={_format_decimal(omnibus['effect'])}$."
        )
        posthoc_name = (
            "paired Student's $t$-tests with Holm correction and "
            "Hedges' $g$ effect sizes"
        )

    leader_mean = descriptive.loc[leader, "Mean Macro-F1"]
    leader_rank = descriptive.loc[leader, "Mean Rank"]

    if pairwise is None:
        pairwise_text = (
            "Because the omnibus result was not significant, "
            "confirmatory pairwise comparisons were not performed."
        )
    else:
        pairwise_text = (
            f"Post-hoc analysis used {posthoc_name}. "
            f"{len(significant)} of {len(pairwise)} pairwise "
            "comparisons remained significant after correction."
        )

    other_leaders = [
        model for model in not_distinguishable if model != leader
    ]
    if other_leaders:
        leader_text = (
            f"**{leader}** had the lowest mean rank "
            f"({leader_rank:.2f}) and the highest corresponding "
            f"descriptive mean Macro-F1 ({leader_mean:.3f}). It was "
            "not statistically distinguishable from: "
            + ", ".join(f"**{model}**" for model in other_leaders)
            + ". This non-significant result does not demonstrate "
            "equivalence."
        )
    else:
        leader_text = (
            f"**{leader}** had the lowest mean rank "
            f"({leader_rank:.2f}) and a mean Macro-F1 of "
            f"{leader_mean:.3f}; it differed significantly from "
            "every other evaluated model."
        )

    if omnibus["significant"]:
        conclusion = (
            "The evaluated approaches did not perform uniformly across "
            "projects. The complete descriptive table should be used "
            "to report all models, while corrected pairwise results "
            "support only the specific statistical distinctions "
            "identified by the post-hoc analysis."
        )
    else:
        conclusion = (
            "The observed model ordering is descriptive because the "
            "omnibus test did not detect an overall difference."
        )

    report = f"""
## {title}

### Assumptions and omnibus test

{assumption_text} {omnibus_text}

### Descriptive ranking and pairwise evidence

{leader_text} {pairwise_text}

### Data-driven interpretation

{conclusion}
"""
    return textwrap.dedent(report)


## Execute the evaluation analysis

Every table and interpretive sentence below is regenerated from the frozen local evaluation CSV. Post-hoc comparisons are performed only when warranted by a significant omnibus test.


In [ ]:
evaluation_analysis = run_model_comparison(df_clean)
evaluation_report = generate_model_comparison_report(
    evaluation_analysis
)
display(Markdown(evaluation_report))


## Article table

The article table is ordered by mean Macro-F1. The complete descriptive, assumption, omnibus, and pairwise outputs are displayed inline by the previous cell. This notebook does not write any files back to `results/`; `metrics/evaluation_macro_f1.csv` remains the only analysis input.


In [ ]:
article_table = (
    evaluation_analysis["summary"][[
        "Mean Macro-F1",
        "SD",
        "Mean Rank",
        "Not distinguishable from leader",
    ]]
    .sort_values("Mean Macro-F1", ascending=False)
    .copy()
)

display(article_table.round({
    "Mean Macro-F1": 3,
    "SD": 3,
    "Mean Rank": 2,
}))


## Holm-corrected pairwise effect-size heatmap

This additional visualization contains only statistically significant pairwise comparisons. It is generated with Matplotlib only; the complete numerical pairwise table displayed above remains the authoritative source for exact values.


In [ ]:
import matplotlib.pyplot as plt

pairwise = evaluation_analysis["pairwise"]

if pairwise is None:
    print(
        "Heatmap not generated because confirmatory post-hoc tests "
        "were not warranted."
    )
else:
    ordered_models = (
        evaluation_analysis["descriptive"]
        .sort_values("Mean Macro-F1", ascending=False)
        .index
        .tolist()
    )
    effect_matrix = pd.DataFrame(
        np.nan,
        index=ordered_models,
        columns=ordered_models,
    )
    p_matrix = effect_matrix.copy()

    for _, row in pairwise.iterrows():
        model_a = row["A"]
        model_b = row["B"]
        effect = float(row["Effect"])
        p_value = float(row["p_Holm"])

        effect_matrix.loc[model_a, model_b] = effect
        effect_matrix.loc[model_b, model_a] = -effect
        p_matrix.loc[model_a, model_b] = p_value
        p_matrix.loc[model_b, model_a] = p_value

    display_matrix = effect_matrix.to_numpy(dtype=float)
    mask = np.ones_like(display_matrix, dtype=bool)
    annotations = {}

    for i in range(len(ordered_models)):
        for j in range(i + 1, len(ordered_models)):
            p_value = p_matrix.iloc[i, j]
            if pd.notna(p_value) and p_value < evaluation_analysis["alpha"]:
                mask[i, j] = False
                annotations[(i, j)] = f"{effect_matrix.iloc[i, j]:.2f}"

    masked = np.ma.masked_where(mask, display_matrix)
    effect_limit = (
        max(1.0, float(pairwise["Absolute effect"].max()))
        if evaluation_analysis["route"] == "parametric"
        else 1.0
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    image = ax.imshow(
        masked,
        vmin=-effect_limit,
        vmax=effect_limit,
        aspect="equal",
    )

    for (i, j), text in annotations.items():
        ax.text(j, i, text, ha="center", va="center")

    ax.set_xticks(range(len(ordered_models)))
    ax.set_yticks(range(len(ordered_models)))
    ax.set_xticklabels(ordered_models, rotation=45, ha="left")
    ax.set_yticklabels(ordered_models)
    ax.xaxis.tick_top()
    ax.set_xlabel("Opponent approach", labelpad=20)
    ax.xaxis.set_label_position("top")
    ax.set_ylabel("Reference approach")
    ax.set_title("Holm-corrected significant pairwise effect sizes", pad=20)

    effect_label = (
        "Hedges' g"
        if evaluation_analysis["route"] == "parametric"
        else "Rank-biserial correlation (r_rb)"
    )
    colorbar = fig.colorbar(image, ax=ax, shrink=0.8)
    colorbar.set_label(effect_label)

    fig.tight_layout()
    plt.show()


## Reproducibility notes

- `metrics/evaluation_macro_f1.csv` is the frozen project-level input to this notebook.
- The notebook requires exactly 30 projects and the six predefined approaches; missing or unexpected data cause an explicit failure.
- Projects are the paired statistical units.
- The route between repeated-measures ANOVA and Friedman is regenerated from the input data according to the prespecified decision rule.
- Mauchly's test and Greenhouse–Geisser correction are used only on the parametric route.
- Confirmatory post-hoc tests are run only after a significant omnibus result.
- Holm correction covers all $k(k-1)/2$ approach pairs.
- Hedges' $g$ is used for paired parametric comparisons; $r_{rb}$ is used for Wilcoxon comparisons.
- Generated tables and the optional heatmap are displayed inline above and can be regenerated at any time by re-running the notebook; nothing is written back to `results/`.
- Scores from incompatible datasets, label spaces, or validation protocols must not be inserted into this paired evaluation.
